In [ ]:
import plotly.graph_objects as go
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df_train["heart_rate"],
    name="Train",
    opacity=0.7
))

fig.add_trace(go.Histogram(
    x=df_test["heart_rate"],
    name="Test",
    opacity=0.7
))

fig.update_layout(
    barmode="overlay",
    title="Heart Rate Distribution",
    xaxis_title="Heart Rate",
    yaxis_title="Count"
)

fig.show()

In [ ]:
import plotly.express as px

fig = px.box(
    df_test,
    x="Label",
    y="heart_rate",
    color="Label",
    title="Heart Rate: Normal vs Anomaly"
)

fig.show()

In [ ]:
fig = px.scatter_matrix(
    df_train[:sample_range],
    dimensions=features_to_plot,
    color="Label",
    title="Feature Relationships"
)

fig.show()

In [ ]:
fig = px.line(
    df_train[:sample_range],
    y="heart_rate",
    title="Heart Rate Over Time"
)

fig.show()

In [ ]:

activity_counts = (
    df_acds.iloc[:,-1]
    .value_counts()
    .reset_index()
)

activity_counts.columns = ["Activity", "Count"]

fig = px.bar(
    activity_counts,
    x="Activity",
    y="Count",
    color="Count",
    title="Number of Samples per Anomaly Activity",
    text="Count"
)

fig.update_layout(xaxis_tickangle=-45)

fig.show()

Scientific explanation: In medicine, a heart rate of 30 beats per minute is a very serious condition (bradycardia), or it may result from the sensor temporarily losing contact with the skin during the anomaly (for example, when the person falls and the device moves, preventing accurate measurement).

Technical explanation: This means that the anomaly in the dataset does not necessarily indicate a “high heart rate”; on the contrary, it could represent a sharp drop or a loss of signal caused by a sudden event.

# Preprocessing

In [ ]:
x_train = df_train.drop(columns=['Label'])
y_train = df_train['Label']

x_test = df_test.drop(columns=['Label'])
y_test = df_test['Label']

del df_train, df_test
gc.collect()

In [ ]:
from sklearn.preprocessing import StandardScaler
input_dir = r'D:\PRIDE_Processed_Folds'

def create_3d_sequences(X_data, y_data, window_size=30):
    X_seq, y_seq = [], []
    for i in range(len(X_data) - window_size):
        X_seq.append(X_data[i : i + window_size])
        y_seq.append(y_data[i + window_size])
    return np.array(X_seq), np.array(y_seq)


for fold in range(1, 6):
    train_path = os.path.join(input_dir, f'fold_{fold}_train.csv')
    test_path = os.path.join(input_dir, f'fold_{fold}_test.csv')
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)

    df_train = df_train.drop_duplicates()

    x_train = df_train.drop(columns=['Label'])
    y_train = df_train['Label']

    x_test = df_test.drop(columns=['Label'])
    y_test = df_test['Label']

    # Isolated forest
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    #LSTM
    x_train_3d, y_train_3d = create_3d_sequences(x_train_scaled, y_train, window_size=30)
    x_test_3d, y_test_3d = create_3d_sequences(x_test_scaled, y_test, window_size=30)

    #Prophet-based لسه

    del df_train, df_test